### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="biomechanical_orthopaedic_prediction",
    dataset_year="2011", # Probably older from WEKA but I cannot find the source...
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5K89B",
    download_description="""
wget https://archive.ics.uci.edu/static/public/212/vertebral+column.zip && unzip vertebral+column.zip column_3C_weka.arff && rm vertebral+column.zip && mkdir -p local-data-warehouse/biomechanical_orthopaedic_prediction && mv column_3C_weka.arff local-data-warehouse/biomechanical_orthopaedic_prediction/
""",
    # References
    academic_reference_bibtex="""@misc{Barreto2005Vertebral,
  author       = {Barreto, Guilherme and Neto, Ajalmar},
  title        = {{Vertebral Column}},
  year         = {2005},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5K89B}
}
""",
    academic_reference_bibtex_key="Barreto2005Vertebral",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the UCI version, which is the oldest existing version of the dataset we found.

-
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import arff

# Read arff file
with open(dataset_mold.path / "column_3C_weka.arff") as f:
        data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

print("Loaded data shape:", df.shape)

as_cat_type = ["class"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (310, 7)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 310
Columns: 7
Use sampling: False (sample size: 310)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['pelvic_incidence', 'pelvic_tilt', 'degree_spondylolisthesis', 'pelvic_radius', 'sacral_slope', 'lumbar_lordosis_angle']
Rows remaining as candidates after top-6 filter: 0 (of 310)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,class
0,44.430701,14.174264,32.243495,30.256437,131.717613,-3.604255,Normal
1,36.686353,5.010884,41.948751,31.675469,84.241415,0.664437,Hernia
2,46.855781,15.351514,38.000000,31.504267,116.250917,1.662706,Hernia
3,74.377678,32.053104,78.772013,42.324573,143.560690,56.125906,Spondylolisthesis
4,54.124920,26.650489,35.329747,27.474432,121.447011,1.571205,Hernia


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,class,category,0.0,0.0,3.0,"Spondylolisthesis, Normal, Hernia"
1,pelvic_incidence,float64,0.0,0.0,310.0,"70.3993, 44.4307, 36.6864, 46.8558, 74.3777, 54.1249, 77.6906, 85.3523, 81.0566, 63.0263"
2,pelvic_tilt,float64,0.0,0.0,310.0,"13.47, 14.1743, 5.0109, 15.3515, 32.0531, 26.6505, 21.3806, 15.8449, 20.8015, 27.3362"
3,lumbar_lordosis_angle,float64,0.0,0.0,280.0,"52.0, 35.0, 47.0, 42.0, 34.0, 58.0, 37.0, 38.0, 43.1992, 39.0"
4,sacral_slope,float64,0.0,0.0,281.0,"35.4171, 56.3099, 45.0, 33.1113, 53.1301, 33.2153, 30.7841, 34.3803, 47.2906, 58.8407"
5,pelvic_radius,float64,0.0,0.0,310.0,"102.3375, 131.7176, 84.2414, 116.2509, 143.5607, 121.447, 114.8188, 124.4198, 125.4302, 114.5066"
6,degree_spondylolisthesis,float64,0.0,0.0,310.0,"25.5384, -3.6043, 0.6644, 1.6627, 56.1259, 1.5712, 26.9318, 76.0206, 38.1818, 7.4399"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
pelvic_incidence,310.0,60.496653,17.236520,26.147921,129.834041
pelvic_tilt,310.0,17.542822,10.008330,-6.554948,49.431864
lumbar_lordosis_angle,310.0,51.930930,18.554064,14.000000,125.742385
sacral_slope,310.0,42.953831,13.423102,13.366931,121.429566
pelvic_radius,310.0,117.920655,13.317377,70.082575,163.071041
degree_spondylolisthesis,310.0,26.296694,37.559027,-11.058179,418.543082


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                                 
class  1     Spondylolisthesis    150  48.39
       2                Normal    100  32.26
       3                Hernia     60  19.35

In [8]:
# Target Distribution
target_df

,count,pct
class,,
Spondylolisthesis,150,48.39
Normal,100,32.26
Hernia,60,19.35


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to biomechanical_orthopaedic_prediction/019d5dc0-1f29-75af-8cb8-52938b3654aa


019d5dc0-1f29-75af-8cb8-52938b3654aa
b5535669441a11b446607a6563cd73a9ab76be7170b1b1eb09379a5746a09eda
